In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

dataset = torchvision.datasets.ImageFolder(
    root='../tiny-imagenet/train',
    transform=transform
)

print("Total images:", len(dataset))
print("Classes:", dataset.classes)

def split_anomaly_dataset(dataset, anomaly_class_idx=9):
    seen_indices = []
    unseen_indices = []

    for i in range(len(dataset)):
        _, label = dataset[i]
        if label == anomaly_class_idx:
            unseen_indices.append(i)
        else:
            seen_indices.append(i)

    seen_dataset = Subset(dataset, seen_indices)
    unseen_dataset = Subset(dataset, unseen_indices)

    return seen_dataset, unseen_dataset

seen_data, unseen_data = split_anomaly_dataset(dataset, anomaly_class_idx=9)

print("Seen samples:", len(seen_data))
print("Unseen samples:", len(unseen_data))

batch_size = 64

seen_loader = DataLoader(seen_data, batch_size=batch_size, shuffle=True)
unseen_loader = DataLoader(unseen_data, batch_size=batch_size, shuffle=False)

Using device: cuda
Total images: 3500
Classes: ['n07871810', 'n07873807', 'n07875152', 'n07920052', 'n09193705', 'n09246464', 'n09256479', 'n09332890', 'n09428293', 'n12267677']
Seen samples: 3150
Unseen samples: 350


In [3]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        
        self.encoder = nn.Sequential(

            # Input: 3 channels (RGB), Output: 16 filters, Kernel: 3x3, Stride: 2
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.ReLU(),

            # Input: 16, Output: 32 filters
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),

            # Input: 32, Output: 64 filters
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

cae_model = ConvAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(cae_model.parameters(), lr=1e-3)

epochs = 30

for epoch in range(epochs):
    cae_model.train()
    running_loss = 0.0

    for images, _ in seen_loader:
        images = images.to(device)
        outputs = cae_model(images)

        loss = criterion(outputs, images)
        optimizer.zero_grad()

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f}")


Epoch [1/30] Loss: 2.8661
Epoch [2/30] Loss: 1.4692
Epoch [3/30] Loss: 1.1899
Epoch [4/30] Loss: 0.9374
Epoch [5/30] Loss: 0.7885
Epoch [6/30] Loss: 0.7220
Epoch [7/30] Loss: 0.6792
Epoch [8/30] Loss: 0.6615
Epoch [9/30] Loss: 0.6419
Epoch [10/30] Loss: 0.6351
Epoch [11/30] Loss: 0.6155
Epoch [12/30] Loss: 0.6088
Epoch [13/30] Loss: 0.5848
Epoch [14/30] Loss: 0.5756
Epoch [15/30] Loss: 0.5648
Epoch [16/30] Loss: 0.5778
Epoch [17/30] Loss: 0.5488
Epoch [18/30] Loss: 0.5435
Epoch [19/30] Loss: 0.5370
Epoch [20/30] Loss: 0.5310
Epoch [21/30] Loss: 0.5288
Epoch [22/30] Loss: 0.5169
Epoch [23/30] Loss: 0.5205
Epoch [24/30] Loss: 0.5030
Epoch [25/30] Loss: 0.5023
Epoch [26/30] Loss: 0.4919
Epoch [27/30] Loss: 0.4893
Epoch [28/30] Loss: 0.4835
Epoch [29/30] Loss: 0.4778
Epoch [30/30] Loss: 0.4698


In [4]:
def evaluate_reconstruction_error(model, dataloader, device):
    model.eval()

    criterion = nn.MSELoss()
    total_loss = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for images, _ in dataloader:
            images = images.to(device)
            outputs = model(images)

            loss = criterion(outputs, images)
            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)
            
    average_mse = total_loss / total_samples
    return average_mse


print("\nEvaluating Seen Classes...")
seen_mse = evaluate_reconstruction_error(cae_model, seen_loader, device)

print("Evaluating Unseen (Anomaly) Class...")
unseen_mse = evaluate_reconstruction_error(cae_model, unseen_loader, device)

print("-" * 30)
print(f"Average MSE for Seen Classes:   {seen_mse:.6f}")
print(f"Average MSE for Unseen Class:   {unseen_mse:.6f}")
print(f"Difference:                    {unseen_mse - seen_mse:.6f}")


Evaluating Seen Classes...
Evaluating Unseen (Anomaly) Class...
------------------------------
Average MSE for Seen Classes:   0.009327
Average MSE for Unseen Class:   0.011276
Difference:                    0.001949


In [6]:
# ==============================
#  ABLATION (Sigmoid vs Tanh final layer)
# ==============================

activations = {
    "Sigmoid": nn.Sigmoid(),
    "Tanh": nn.Tanh()
}

ablation_results = {}

for name, final_act in activations.items():
    model = ConvAutoencoder().to(device)
    model.decoder[-1] = final_act 

    print(f"\nRunning with output layer = {name}")
    print(f"Final decoder layer now: {model.decoder[-1]}")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for _ in range(epochs):
        model.train()
        for images, _ in seen_loader:
            images = images.to(device)
            outputs = model(images)
            loss = criterion(outputs, images)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    seen_mse = evaluate_reconstruction_error(model, seen_loader, device)
    unseen_mse = evaluate_reconstruction_error(model, unseen_loader, device)
    gap = unseen_mse - seen_mse

    ablation_results[name] = {
        "seen_mse": seen_mse,
        "unseen_mse": unseen_mse,
        "gap": gap
    }

    print(f"Seen MSE:   {seen_mse:.6f}")
    print(f"Unseen MSE: {unseen_mse:.6f}")
    print(f"Gap:        {gap:.6f}")

print("\n" + "=" * 45)
print("Comparison Summary")
print("=" * 45)

for name, metrics in ablation_results.items():
    print(f"{name:8s} | Seen: {metrics['seen_mse']:.6f} | Unseen: {metrics['unseen_mse']:.6f} | Gap: {metrics['gap']:.6f}")

best_seen = min(ablation_results, key=lambda k: ablation_results[k]["seen_mse"])
best_gap = max(ablation_results, key=lambda k: ablation_results[k]["gap"])

print(f"\nLower reconstruction loss on seen data: {best_seen}")
print(f"Higher anomaly separation (gap):       {best_gap}")


Running with output layer = Sigmoid
Final decoder layer now: Sigmoid()
Seen MSE:   0.008731
Unseen MSE: 0.010526
Gap:        0.001795

Running with output layer = Tanh
Final decoder layer now: Tanh()
Seen MSE:   0.008108
Unseen MSE: 0.008521
Gap:        0.000413

Comparison Summary
Sigmoid  | Seen: 0.008731 | Unseen: 0.010526 | Gap: 0.001795
Tanh     | Seen: 0.008108 | Unseen: 0.008521 | Gap: 0.000413

Lower reconstruction loss on seen data: Tanh
Higher anomaly separation (gap):       Sigmoid
